# Cognix Round 2: Analysis

**Runs locally on Mac (no GPU needed).** Uses cached vectors from the Colab inference run.

**What this tests:**
1. Does the Round 1 divergence hold at 1,000 pairs?
2. **Does TRIBE add value beyond raw LLaMA 3.2 embeddings?** (the critical test)
3. Does mean-centering fix the high baseline problem?
4. What is the random baseline brain similarity?
5. Per-category statistical significance with 25+ pairs each
6. Overfitting checks: text-length correlation, adversarial pairs, permutation tests

In [ ]:
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path

In [ ]:
# ---- Config ----
# Adjust these paths based on where you put the cached vectors
PAIRS_PATH = '../data/validation_pairs_r2.jsonl'
POOLED_DIR = Path('../artifacts/r2_pooled_vectors')   # TRIBE brain vectors from Google Drive
LLAMA_DIR = Path('../artifacts/r2_llama_vectors')      # LLaMA 3.2-3B embeddings from Google Drive

def text_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

## 1. Load pairs and cached brain vectors

In [ ]:
pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))

print(f'Total pairs: {len(pairs)}')
sources = {}
for p in pairs:
    src = p['source']
    sources[src] = sources.get(src, 0) + 1
print('Sources:', sources)

categories = {}
for p in pairs:
    cat = p['category']
    categories[cat] = categories.get(cat, 0) + 1
print('Categories:', dict(sorted(categories.items())))

In [ ]:
# Load all cached pooled vectors into a dict
brain_cache = {}
for npy_file in POOLED_DIR.glob('*.npy'):
    h = npy_file.stem
    brain_cache[h] = np.load(npy_file)
print(f'Loaded {len(brain_cache)} cached brain vectors')

## 2. Compute semantic similarity (sentence-transformer)

In [ ]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')

semantic_sims = []
for p in tqdm(pairs, desc='Semantic similarity'):
    emb_a = st_model.encode(p['text_a'])
    emb_b = st_model.encode(p['text_b'])
    sim = 1.0 - cosine_dist(emb_a, emb_b)
    semantic_sims.append(sim)
semantic_sims = np.array(semantic_sims)

## 3. Compute brain similarity (from cached vectors)

In [ ]:
brain_sims = []
valid_mask = []
for p in pairs:
    ha = text_hash(p['text_a'])
    hb = text_hash(p['text_b'])
    va = brain_cache.get(ha)
    vb = brain_cache.get(hb)
    if va is not None and vb is not None:
        sim = 1.0 - cosine_dist(va, vb)
        brain_sims.append(sim)
        valid_mask.append(True)
    else:
        brain_sims.append(np.nan)
        valid_mask.append(False)

brain_sims = np.array(brain_sims)
valid_mask = np.array(valid_mask)
print(f'Brain similarity computed for {valid_mask.sum()}/{len(pairs)} pairs')

## 4. LLaMA 3.2-3B baseline (THE CRITICAL TEST)

TRIBE's text pathway feeds text through LLaMA 3.2-3B, then maps those features to brain activity. If raw LLaMA embeddings show the same divergence pattern as brain vectors, the brain mapping adds nothing — we're just re-encoding LLaMA in a noisier space.

This loads the cached LLaMA 3.2-3B embeddings (last hidden state, mean-pooled) extracted during the inference run.

In [ ]:
# Load cached LLaMA 3.2-3B embeddings
llama_cache = {}
for npy_file in LLAMA_DIR.glob('*.npy'):
    h = npy_file.stem
    llama_cache[h] = np.load(npy_file)
print(f'Loaded {len(llama_cache)} cached LLaMA vectors')

if len(llama_cache) == 0:
    print('\nWARNING: No LLaMA vectors found!')
    print(f'Expected directory: {LLAMA_DIR.resolve()}')
    print('Run the LLaMA extraction cells in 01_r2_tribe_inference.ipynb first,')
    print('then download llama_vectors/ from Google Drive to artifacts/r2_llama_vectors/')

# Compute LLaMA similarity for each pair
llama_sims = []
llama_valid = []
for p in pairs:
    ha = text_hash(p['text_a'])
    hb = text_hash(p['text_b'])
    va = llama_cache.get(ha)
    vb = llama_cache.get(hb)
    if va is not None and vb is not None:
        sim = 1.0 - cosine_dist(va, vb)
        llama_sims.append(sim)
        llama_valid.append(True)
    else:
        llama_sims.append(np.nan)
        llama_valid.append(False)

llama_sims = np.array(llama_sims)
llama_valid = np.array(llama_valid)
print(f'LLaMA similarity computed for {llama_valid.sum()}/{len(pairs)} pairs')
if llama_valid.sum() > 0:
    print(f'LLaMA similarity: mean={np.nanmean(llama_sims):.3f}, std={np.nanstd(llama_sims):.3f}')

## 5. Three-way correlation — the key comparison

The critical question: does brain sim ≠ LLaMA sim? If so, the brain mapping reshapes the geometry.

In [ ]:
# ---- Three-way correlation: semantic vs LLaMA vs brain ----
# Use pairs where all three metrics are valid
all_valid = valid_mask & llama_valid

print('=' * 70)
print('THREE-WAY CORRELATION (all pairs where all metrics are available)')
print('=' * 70)

if all_valid.sum() > 0:
    sem_v = semantic_sims[all_valid]
    brain_v = brain_sims[all_valid]
    llama_v = llama_sims[all_valid]

    # Brain vs Semantic
    r1, p1 = stats.pearsonr(sem_v, brain_v)
    s1, sp1 = stats.spearmanr(sem_v, brain_v)
    print(f'\n  Brain vs Semantic (n={all_valid.sum()})')
    print(f'    Pearson r:  {r1:.4f}  (p={p1:.2e})')
    print(f'    Spearman r: {s1:.4f}  (p={sp1:.2e})')

    # Brain vs LLaMA
    r2, p2 = stats.pearsonr(llama_v, brain_v)
    s2, sp2 = stats.spearmanr(llama_v, brain_v)
    print(f'\n  Brain vs LLaMA (n={all_valid.sum()})')
    print(f'    Pearson r:  {r2:.4f}  (p={p2:.2e})')
    print(f'    Spearman r: {s2:.4f}  (p={sp2:.2e})')

    # LLaMA vs Semantic
    r3, p3 = stats.pearsonr(sem_v, llama_v)
    s3, sp3 = stats.spearmanr(sem_v, llama_v)
    print(f'\n  LLaMA vs Semantic (n={all_valid.sum()})')
    print(f'    Pearson r:  {r3:.4f}  (p={p3:.2e})')
    print(f'    Spearman r: {s3:.4f}  (p={sp3:.2e})')

    print('\n' + '=' * 70)
    print('INTERPRETATION')
    print('=' * 70)
    if r2 > 0.8:
        print(f'  Brain vs LLaMA r={r2:.2f} — HIGH correlation.')
        print('  The brain mapping may not be adding much beyond LLaMA features.')
        print('  The divergence from semantic similarity is likely just LLaMA ≠ sentence-transformers.')
    elif r2 > 0.5:
        print(f'  Brain vs LLaMA r={r2:.2f} — MODERATE correlation.')
        print('  The brain mapping partially reshapes the geometry. Check per-category differences.')
    else:
        print(f'  Brain vs LLaMA r={r2:.2f} — LOW correlation.')
        print('  The brain mapping substantially reshapes the similarity geometry.')
        print('  TRIBE adds value beyond what LLaMA already encodes.')
else:
    print('  No pairs with all three metrics available.')
    print('  Make sure LLaMA vectors are cached and loaded.')

# Also show per-source breakdown
print('\n' + '=' * 70)
print('BRAIN vs SEMANTIC — per source')
print('=' * 70)
for source_name in ['handcrafted', 'stsb', 'wikipedia_random']:
    mask = np.array([p['source'] == source_name and valid_mask[i] for i, p in enumerate(pairs)])
    if mask.sum() < 5:
        continue
    r_p, p_p = stats.pearsonr(semantic_sims[mask], brain_sims[mask])
    print(f'  {source_name} (n={mask.sum()}): Pearson r={r_p:.4f} (p={p_p:.2e})')

## 6. Random baseline distribution (quantifies the high-baseline problem)

In [ ]:
random_mask = np.array([p['source'] == 'wikipedia_random' and valid_mask[i] for i, p in enumerate(pairs)])
random_brain = brain_sims[random_mask]
random_sem = semantic_sims[random_mask]

print('RANDOM WIKIPEDIA PAIRS (baseline brain similarity)')
print(f'  N: {random_mask.sum()}')
print(f'  Brain sim:    mean={np.nanmean(random_brain):.3f}, std={np.nanstd(random_brain):.3f}')
print(f'  Semantic sim: mean={np.nanmean(random_sem):.3f}, std={np.nanstd(random_sem):.3f}')
print()
print(f'  This is the "floor" — the brain similarity you get for arbitrary text pairs.')
print(f'  Any divergence category must score ABOVE this to be meaningful.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(random_brain[~np.isnan(random_brain)], bins=30, alpha=0.7, label='Brain sim (random pairs)')
ax.axvline(np.nanmean(random_brain), color='red', linestyle='--', label=f'Mean: {np.nanmean(random_brain):.3f}')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.set_title('Brain similarity distribution for random Wikipedia pairs')
ax.legend()
plt.tight_layout()
plt.show()

## 6b. Baseline removal: mean-centering

Mean-pooled brain vectors average ~0.82 cosine similarity for unrelated pairs. Subtracting the corpus-average vector should compress this floor toward 0 and reveal the true dynamic range.

In [ ]:
# ---- Mean-center brain vectors and recompute similarity ----
if len(brain_cache) > 0:
    # Compute corpus mean from all cached brain vectors
    all_brain_vecs = np.stack(list(brain_cache.values()))
    corpus_mean = all_brain_vecs.mean(axis=0)
    print(f'Corpus mean computed from {len(brain_cache)} brain vectors')
    print(f'Corpus mean norm: {np.linalg.norm(corpus_mean):.2f}')

    # Mean-center and recompute similarity
    brain_sims_centered = []
    for p in pairs:
        ha = text_hash(p['text_a'])
        hb = text_hash(p['text_b'])
        va = brain_cache.get(ha)
        vb = brain_cache.get(hb)
        if va is not None and vb is not None:
            va_c = va - corpus_mean
            vb_c = vb - corpus_mean
            sim = 1.0 - cosine_dist(va_c, vb_c)
            brain_sims_centered.append(sim)
        else:
            brain_sims_centered.append(np.nan)

    brain_sims_centered = np.array(brain_sims_centered)

    # Compare before/after
    print(f'\nBefore mean-centering:')
    print(f'  Brain sim (all valid): mean={np.nanmean(brain_sims):.3f}, std={np.nanstd(brain_sims):.3f}')
    print(f'  Random pairs:          mean={np.nanmean(random_brain):.3f}')

    random_brain_centered = brain_sims_centered[random_mask]
    print(f'\nAfter mean-centering:')
    print(f'  Brain sim (all valid): mean={np.nanmean(brain_sims_centered):.3f}, std={np.nanstd(brain_sims_centered):.3f}')
    print(f'  Random pairs:          mean={np.nanmean(random_brain_centered):.3f}')

    # Re-check correlation with mean-centered brain sims
    centered_valid = valid_mask & ~np.isnan(brain_sims_centered)
    if centered_valid.sum() > 0:
        r_c, p_c = stats.pearsonr(semantic_sims[centered_valid], brain_sims_centered[centered_valid])
        print(f'\n  Pearson r (semantic vs centered brain): {r_c:.4f} (p={p_c:.2e})')
else:
    print('No brain vectors loaded — skipping mean-centering.')

## 7. Per-category breakdown — all four metrics

The key table. Shows semantic, LLaMA, brain, and centered brain similarity per category. The critical comparison is **Brain vs LLaMA**: if they're close, the brain mapping isn't adding value for that category.

In [ ]:
handcrafted_cats = sorted(set(
    p['category'] for p in pairs if p['source'] == 'handcrafted'
))

# ---- Table 1: All four metrics per category ----
print('ALL FOUR METRICS PER CATEGORY (handcrafted pairs)')
print(f'{"Category":<35} {"N":>3}  {"Sem":>6}  {"LLaMA":>6}  {"Brain":>6}  {"Centered":>8}')
print('-' * 80)

for cat in handcrafted_cats:
    mask = np.array([
        p['category'] == cat and p['source'] == 'handcrafted' and valid_mask[i]
        for i, p in enumerate(pairs)
    ])
    if mask.sum() < 3:
        continue

    sem = semantic_sims[mask].mean()
    brain = brain_sims[mask].mean()

    # LLaMA (may not be available for all pairs)
    llama_mask = mask & llama_valid
    llama_str = f'{llama_sims[llama_mask].mean():>6.3f}' if llama_mask.sum() > 0 else '   N/A'

    # Centered brain
    centered_mask = mask & ~np.isnan(brain_sims_centered)
    cent_str = f'{brain_sims_centered[centered_mask].mean():>8.3f}' if centered_mask.sum() > 0 else '     N/A'

    print(f'{cat:<35} {mask.sum():>3}  {sem:>6.3f}  {llama_str}  {brain:>6.3f}  {cent_str}')

# ---- Table 2: Brain vs LLaMA divergence per category ----
print()
print('BRAIN vs LLaMA DIVERGENCE PER CATEGORY')
print('(Positive = brain sees MORE similarity than LLaMA)')
print(f'{"Category":<35} {"N":>3}  {"LLaMA":>6}  {"Brain":>6}  {"Diff":>7}')
print('-' * 65)

for cat in handcrafted_cats:
    mask = np.array([
        p['category'] == cat and p['source'] == 'handcrafted' and valid_mask[i] and llama_valid[i]
        for i, p in enumerate(pairs)
    ])
    if mask.sum() < 3:
        continue

    llama_m = llama_sims[mask].mean()
    brain_m = brain_sims[mask].mean()
    diff = brain_m - llama_m
    print(f'{cat:<35} {mask.sum():>3}  {llama_m:>6.3f}  {brain_m:>6.3f}  {diff:>+7.3f}')

# ---- Table 3: Paired t-test brain vs semantic (original) ----
print()
print('SIGNIFICANCE: Brain vs Semantic per category')
print(f'{"Category":<35} {"N":>3}  {"Sem":>6}  {"Brain":>6}  {"Diff":>7}  {"p-value":>8}')
print('-' * 80)

for cat in handcrafted_cats:
    mask = np.array([
        p['category'] == cat and p['source'] == 'handcrafted' and valid_mask[i]
        for i, p in enumerate(pairs)
    ])
    if mask.sum() < 3:
        continue

    sem = semantic_sims[mask]
    brain = brain_sims[mask]
    diff = brain - sem
    t_stat, p_val = stats.ttest_rel(brain, sem)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f'{cat:<35} {mask.sum():>3}  {sem.mean():>6.3f}  {brain.mean():>6.3f}  {diff.mean():>+7.3f}  {p_val:>8.2e} {sig}')

## 8. Scatter plots — raw and centered

Side-by-side comparison. The centered plot should show better separation if mean-centering fixes the baseline compression.

In [ ]:
category_colors = {
    'paraphrase': '#2ecc71',
    'unrelated': '#95a5a6',
    'cognitive_load': '#e74c3c',
    'emotional_arousal': '#e67e22',
    'sensorimotor': '#9b59b6',
    'spatial_scene': '#3498db',
    'syntactic_complexity': '#1abc9c',
    'narrative_suspense': '#f39c12',
    'theory_of_mind': '#e91e63',
    'control_same_topic_diff_processing': '#607d8b',
    'adversarial': '#000000',
    'stsb': '#bdc3c7',
    'random_baseline': '#dfe6e9',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax_idx, (brain_data, title_suffix, fname_suffix) in enumerate([
    (brain_sims, 'Raw Brain Similarity', 'raw'),
    (brain_sims_centered, 'Centered Brain Similarity', 'centered'),
]):
    ax = axes[ax_idx]

    for cat in category_colors:
        mask = np.array([p['category'] == cat and valid_mask[i] for i, p in enumerate(pairs)])
        if mask.sum() == 0:
            continue
        y_data = brain_data[mask]
        good = ~np.isnan(y_data)
        if good.sum() == 0:
            continue
        ax.scatter(
            semantic_sims[mask][good], y_data[good],
            c=category_colors[cat], label=cat, alpha=0.6, s=30,
            edgecolors='white', linewidth=0.3,
        )

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x')

    bd_valid = ~np.isnan(brain_data) & valid_mask
    if bd_valid.sum() > 0:
        r_p, _ = stats.pearsonr(semantic_sims[bd_valid], brain_data[bd_valid])
        r_s, _ = stats.spearmanr(semantic_sims[bd_valid], brain_data[bd_valid])
    else:
        r_p, r_s = 0, 0

    ax.set_xlabel('Semantic Similarity', fontsize=12)
    ax.set_ylabel(title_suffix, fontsize=12)
    ax.set_title(f'Cognix Round 2: Semantic vs {title_suffix}\nPearson r={r_p:.3f}, Spearman r={r_s:.3f}', fontsize=13)
    ax.legend(loc='upper left', fontsize=6, ncol=2)
    ax.set_xlim(-0.15, 1.05)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('r2_scatter.png', dpi=150)
plt.show()

## 9. Overfitting checks

In [ ]:
# ---- Check 1: Text length correlation ----
# Is brain similarity just a proxy for text length?

len_a = np.array([len(p['text_a'].split()) for p in pairs])
len_b = np.array([len(p['text_b'].split()) for p in pairs])
len_diff = np.abs(len_a - len_b)
len_mean = (len_a + len_b) / 2

mask_v = valid_mask
r_len_brain, p_len_brain = stats.pearsonr(len_mean[mask_v], brain_sims[mask_v])
r_lendiff_brain, p_lendiff_brain = stats.pearsonr(len_diff[mask_v], brain_sims[mask_v])

print('TEXT LENGTH vs BRAIN SIMILARITY')
print(f'  Mean text length vs brain sim:  r={r_len_brain:.3f}  (p={p_len_brain:.2e})')
print(f'  Length difference vs brain sim: r={r_lendiff_brain:.3f}  (p={p_lendiff_brain:.2e})')
print()
if abs(r_len_brain) > 0.5:
    print('  WARNING: Strong correlation between text length and brain similarity.')
    print('  The brain signal may be confounded with text length.')
else:
    print('  Text length is not strongly correlated with brain similarity. Good.')

In [ ]:
# ---- Check 2: Adversarial pairs ----
adv_mask = np.array([p['category'] == 'adversarial' and valid_mask[i] for i, p in enumerate(pairs)])

if adv_mask.sum() > 0:
    print('ADVERSARIAL PAIRS')
    print(f'  N: {adv_mask.sum()}')
    for i, p in enumerate(pairs):
        if p['category'] == 'adversarial' and valid_mask[i]:
            print(f'  id={p["id"]}  expected={p["expected"]}  sem={semantic_sims[i]:.3f}  brain={brain_sims[i]:.3f}')
            print(f'    A: {p["text_a"][:80]}...')
            print(f'    B: {p["text_b"][:80]}...')
            print()

In [ ]:
# ---- Check 3: Permutation test ----
# Shuffle category labels and recompute per-category divergence.
# If the real divergence is within the shuffled distribution, it's not meaningful.

handcrafted_mask = np.array([p['source'] == 'handcrafted' and valid_mask[i] for i, p in enumerate(pairs)])
hc_brain = brain_sims[handcrafted_mask]
hc_sem = semantic_sims[handcrafted_mask]
hc_cats = [p['category'] for i, p in enumerate(pairs) if p['source'] == 'handcrafted' and valid_mask[i]]

# Real mean divergence for divergence categories
divergence_cats = {'cognitive_load', 'emotional_arousal', 'sensorimotor', 'spatial_scene',
                   'syntactic_complexity', 'narrative_suspense', 'theory_of_mind'}
div_mask = np.array([c in divergence_cats for c in hc_cats])
real_div_mean = (hc_brain[div_mask] - hc_sem[div_mask]).mean()

# Permutation: shuffle categories 1000 times
rng = np.random.default_rng(42)
n_perms = 1000
perm_divs = []
for _ in range(n_perms):
    shuffled = rng.permutation(hc_cats)
    perm_div_mask = np.array([c in divergence_cats for c in shuffled])
    perm_div = (hc_brain[perm_div_mask] - hc_sem[perm_div_mask]).mean()
    perm_divs.append(perm_div)

perm_divs = np.array(perm_divs)
p_perm = (perm_divs >= real_div_mean).mean()

print('PERMUTATION TEST')
print(f'  Real mean divergence (divergence categories): {real_div_mean:.3f}')
print(f'  Permuted mean divergence: mean={perm_divs.mean():.3f}, std={perm_divs.std():.3f}')
print(f'  Permutation p-value: {p_perm:.4f}')
if p_perm < 0.05:
    print('  The divergence is significant beyond what shuffled labels would produce.')
else:
    print('  WARNING: The divergence is not significant under permutation testing.')

## 10. Top 30 most divergent pairs

In [ ]:
divergence = brain_sims - semantic_sims
valid_indices = np.where(valid_mask)[0]
sorted_idx = valid_indices[np.argsort(divergence[valid_indices])[::-1]]

print('TOP 30: Brain says similar, semantics says not')
print('=' * 90)
for rank, idx in enumerate(sorted_idx[:30], 1):
    p = pairs[idx]
    print(f'\n#{rank} (id={p["id"]}, src={p["source"]}, cat={p["category"]})')
    print(f'  Sem: {semantic_sims[idx]:.3f}  Brain: {brain_sims[idx]:.3f}  Div: {divergence[idx]:+.3f}')
    print(f'  A: {p["text_a"][:100]}...')
    print(f'  B: {p["text_b"][:100]}...')

## 11. Summary

In [ ]:
mask_all = valid_mask
r_p, p_p = stats.pearsonr(semantic_sims[mask_all], brain_sims[mask_all])
r_s, p_s = stats.spearmanr(semantic_sims[mask_all], brain_sims[mask_all])

print('ROUND 2 SUMMARY')
print('=' * 60)
print(f'Total pairs: {len(pairs)}')
print(f'Valid pairs (brain): {valid_mask.sum()}')
print(f'Valid pairs (LLaMA): {llama_valid.sum()}')
print()
print(f'Pearson r  (brain vs semantic): {r_p:.4f}  (p={p_p:.2e})')
print(f'Spearman r (brain vs semantic): {r_s:.4f}  (p={p_s:.2e})')

# LLaMA baseline result
all_v = valid_mask & llama_valid
if all_v.sum() > 0:
    r_bl, p_bl = stats.pearsonr(llama_sims[all_v], brain_sims[all_v])
    r_ls, p_ls = stats.pearsonr(semantic_sims[all_v], llama_sims[all_v])
    print(f'Pearson r  (brain vs LLaMA):    {r_bl:.4f}  (p={p_bl:.2e})')
    print(f'Pearson r  (LLaMA vs semantic): {r_ls:.4f}  (p={p_ls:.2e})')

print()
print(f'Random baseline brain sim: {np.nanmean(random_brain):.3f} +/- {np.nanstd(random_brain):.3f}')
print(f'Text-length correlation:   r={r_len_brain:.3f}')
print(f'Permutation test p-value:  {p_perm:.4f}')

print()
print('KEY QUESTIONS ANSWERED:')
print(f'  1. Does divergence hold at scale?  r={r_p:.2f} (compare to Round 1 r=0.24)')
print(f'  2. Does brain ≠ LLaMA?             r={r_bl:.2f}' if all_v.sum() > 0 else '  2. Does brain ≠ LLaMA?             NO LLAMA DATA')
print(f'  3. Permutation significant?         p={p_perm:.4f}')

In [ ]:
# ---- Save results ----
results = []
for i, p in enumerate(pairs):
    results.append({
        'id': p['id'],
        'source': p['source'],
        'category': p['category'],
        'expected': p['expected'],
        'sim_semantic': float(semantic_sims[i]),
        'sim_brain': float(brain_sims[i]) if valid_mask[i] else None,
        'sim_llama': float(llama_sims[i]) if llama_valid[i] else None,
        'sim_brain_centered': float(brain_sims_centered[i]) if (valid_mask[i] and not np.isnan(brain_sims_centered[i])) else None,
        'text_a_len': len(p['text_a'].split()),
        'text_b_len': len(p['text_b'].split()),
    })

with open('../data/r2_results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print('Results saved to data/r2_results.jsonl')